# Data Quality Analysis

**Dataset columns:**
- `tranaction_id` — unique transaction identifier
- `tx_datetime` — transaction timestamp
- `customer_id` — customer identifier
- `terminal_id` — terminal identifier
- `tx_amount` — transaction amount
- `tx_time_seconds` — transaction time in seconds from epoch
- `tx_time_days` — transaction time in days from epoch
- `tx_fraud` — fraud label (0/1)
- `tx_fraud_scenario` — fraud scenario type (0-3)

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("data-quality-analysis") \
    .getOrCreate()

BUCKET_NAME = "otus-mlops-data-b1g7vl7oovirupu7q3vf"

schema = StructType([
    StructField("tranaction_id", LongType(), True),
    StructField("tx_datetime", StringType(), True),
    StructField("customer_id", LongType(), True),
    StructField("terminal_id", StringType(), True),
    StructField("tx_amount", DoubleType(), True),
    StructField("tx_time_seconds", LongType(), True),
    StructField("tx_time_days", LongType(), True),
    StructField("tx_fraud", IntegerType(), True),
    StructField("tx_fraud_scenario", IntegerType(), True),
])

df = spark.read.csv(
    # для скорости взят один файл
    f"s3a://{BUCKET_NAME}/2022-11-04.txt",
    # по всем файлам
    # f"s3a://{BUCKET_NAME}/*.txt", 
    schema=schema,
    header=False,
    comment="#",
)

print(f"Total rows: {df.count()}")
df.printSchema()
df.show(5, truncate=False)

Total rows: 46998983
root
 |-- tranaction_id: long (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: long (nullable = true)
 |-- tx_time_days: long (nullable = true)
 |-- tx_fraud: integer (nullable = true)
 |-- tx_fraud_scenario: integer (nullable = true)

+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|tranaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|1832792610   |2022-11-04 14:22:18|0          |53         |63.58    |101139738      |1170        |0       |0                |
|1832792611   |2022-11-04 02:12:24|0          |53         |92.95    |101095944  

## 1. Missing Values (NULL)

Пропуски в колонках

In [4]:
null_counts = df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
)
null_counts.show(truncate=False)

print("Sample rows with NULL terminal_id:")
df.filter(col("terminal_id").isNull()).show(10, truncate=False)

+-------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|tranaction_id|tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+-------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|0            |0          |0          |85         |0        |0              |0           |0       |0                |
+-------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+

Sample rows with NULL terminal_id:


[Stage 12:=============================>                            (2 + 2) / 4]

+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|tranaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|1833137098   |2022-11-04 05:30:54|220722     |null       |126.99   |101107854      |1170        |0       |0                |
|1833776547   |2022-11-04 08:46:39|628891     |null       |6.79     |101119599      |1170        |0       |0                |
|1834518761   |2022-11-05 09:28:18|102444     |null       |53.53    |101208498      |1171        |0       |0                |
|1834874213   |2022-11-05 20:39:12|329494     |null       |23.65    |101248752      |1171        |0       |0                |
|1834918073   |2022-11-05 14:53:15|357362     |null       |49.91    |101227995      |1171        |0       |0          

## 2. Invalid Values in terminal_id

terminal_id должна быть numeric, но некоторые колоник содержат 'Err' строку.

In [5]:
df_with_terminal_check = df.withColumn(
    "terminal_id_numeric",
    col("terminal_id").cast(IntegerType())
)

non_numeric_terminal = df_with_terminal_check.filter(
    col("terminal_id").isNotNull() & col("terminal_id_numeric").isNull()
)

print(f"Non-numeric terminal_id count: {non_numeric_terminal.count()}")
print("\nDistinct invalid terminal_id values:")
non_numeric_terminal.select("terminal_id").distinct().show()
print("Sample rows:")
non_numeric_terminal.show(10, truncate=False)

[Stage 13:=====================================================>  (23 + 1) / 24]

Non-numeric terminal_id count: 2213

Distinct invalid terminal_id values:


+-----------+
|terminal_id|
+-----------+
|        Err|
+-----------+

Sample rows:


[Stage 25:>                                                         (0 + 1) / 1]

+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+-------------------+
|tranaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|terminal_id_numeric|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+-------------------+
|1832799975   |2022-11-04 13:13:25|4776       |Err        |5.21     |101135605      |1170        |1       |2                |null               |
|1832820886   |2022-11-04 08:46:02|18099      |Err        |96.56    |101119562      |1170        |0       |0                |null               |
|1832891084   |2022-11-04 07:26:02|62904      |Err        |34.99    |101114762      |1170        |0       |0                |null               |
|1832891819   |2022-11-04 11:49:27|63351      |Err        |60.33    |101130567      |1170        |0       |0                

## 3. Negative customer_id

customer_id должна быть неотрицательна, но некоторые строки содержать отрицательный значения.

In [6]:
negative_customers = df.filter(col("customer_id") < 0)
print(f"Negative customer_id count: {negative_customers.count()}")
print("\nSample rows:")
negative_customers.show(10, truncate=False)
print("\nDistinct negative customer_id values:")
negative_customers.select("customer_id").distinct().show()

Negative customer_id count: 96

Sample rows:


+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|tranaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|1834231246   |2022-11-04 06:09:28|-999999    |168        |77.45    |101110168      |1170        |0       |0                |
|1834432304   |2022-11-05 19:13:44|-999999    |951        |53.97    |101243624      |1171        |0       |0                |
|1834802463   |2022-11-05 16:19:28|-999999    |716        |3.89     |101233168      |1171        |0       |0                |
|1835071499   |2022-11-05 08:27:38|-999999    |433        |66.67    |101204858      |1171        |0       |0                |
|1835640422   |2022-11-05 20:11:08|-999999    |331        |20.15    |101247068      |1171        |0       |0          

+-----------+
|customer_id|
+-----------+
|    -999999|
+-----------+



## 4. Invalid Dates (24:00:00)

У некоторых таймстемпов hour=24, это невалидное значение.

In [7]:
# Dates with 24:00:00 — Spark's to_timestamp will return NULL for these
df_with_ts = df.withColumn("parsed_ts", to_timestamp(col("tx_datetime")))
invalid_dates = df_with_ts.filter(
    col("tx_datetime").isNotNull() & col("parsed_ts").isNull()
)

print(f"Invalid date count: {invalid_dates.count()}")
print("\nSample rows:")
invalid_dates.select("tranaction_id", "tx_datetime").show(10, truncate=False)

Invalid date count: 92

Sample rows:


[Stage 43:>                                                         (0 + 2) / 2]

+-------------+-------------------+
|tranaction_id|tx_datetime        |
+-------------+-------------------+
|1832907535   |2022-11-04 24:00:00|
|1833166751   |2022-11-04 24:00:00|
|1833439501   |2022-11-04 24:00:00|
|1833859309   |2022-11-04 24:00:00|
|1834587620   |2022-11-05 24:00:00|
|1834874914   |2022-11-05 24:00:00|
|1835555154   |2022-11-05 24:00:00|
|1835631794   |2022-11-05 24:00:00|
|1835749582   |2022-11-05 24:00:00|
|1836236131   |2022-11-06 24:00:00|
+-------------+-------------------+
only showing top 10 rows



## 5. Zero Transaction Amount

Транзакции с размером транзакции=0 - скорее всего ошибка.

In [8]:
zero_amount = df.filter(col("tx_amount") == 0)
print(f"Zero amount count: {zero_amount.count()}")
print("\nSample rows:")
zero_amount.show(10, truncate=False)

Zero amount count: 936

Sample rows:


[Stage 46:>                                                         (0 + 1) / 1]

+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|tranaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|1832822690   |2022-11-04 10:44:02|19222      |0          |0.0      |101126642      |1170        |0       |0                |
|1832932161   |2022-11-04 11:29:56|89294      |902        |0.0      |101129396      |1170        |0       |0                |
|1832939642   |2022-11-04 19:34:56|94090      |485        |0.0      |101158496      |1170        |0       |0                |
|1832967701   |2022-11-04 06:29:45|112206     |275        |0.0      |101111385      |1170        |0       |0                |
|1832972915   |2022-11-04 14:46:36|115539     |65         |0.0      |101141196      |1170        |0       |0          

## 6. Duplicate Transaction IDs

In [13]:
from pyspark.sql.window import Window

w = Window.partitionBy("tranaction_id")
df_dup = df.withColumn("dup_count", count("*").over(w))
duplicates = df_dup.filter(col("dup_count") > 1)

print(f"Rows with duplicate transaction IDs: {duplicates.count()}")
print("\nDuplicate rows:")
duplicates.orderBy("tranaction_id").show(20, truncate=False)

Rows with duplicate transaction IDs: 16

Duplicate rows:


[Stage 84:===================================================>  (191 + 6) / 200]

+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+---------+
|tranaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|dup_count|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+---------+
|1838630470   |2022-11-07 18:41:37|726029     |145        |19.78    |101414497      |1173        |0       |0                |2        |
|1838630470   |2022-11-07 18:41:37|726029     |145        |19.78    |101414497      |1173        |0       |0                |2        |
|1839244215   |2022-11-08 04:52:33|118317     |882        |12.84    |101451153      |1174        |0       |0                |2        |
|1839244215   |2022-11-08 04:52:33|118317     |882        |12.84    |101451153      |1174        |0       |0                |2        |
|1843366603   |2022-11-10 12:14:09|749945     |5

## 7. Basic Statistics

In [12]:
df.describe().show(truncate=False)

# Fraud distribution
print("\nFraud distribution:")
df.groupBy("tx_fraud").count().show()

print("\nFraud scenario distribution:")
df.groupBy("tx_fraud_scenario").count().orderBy("tx_fraud_scenario").show()

+-------+-------------------+-------------------+-----------------+------------------+-----------------+--------------------+------------------+-------------------+--------------------+
|summary|tranaction_id      |tx_datetime        |customer_id      |terminal_id       |tx_amount        |tx_time_seconds     |tx_time_days      |tx_fraud           |tx_fraud_scenario   |
+-------+-------------------+-------------------+-----------------+------------------+-----------------+--------------------+------------------+-------------------+--------------------+
|count  |46998983           |46998983           |46998983         |46998898          |46998983         |46998983            |46998983          |46998983           |46998983            |
|mean   |1.856292096624556E9|null               |500437.420257349 |28085.477290004605|54.17185979556221|1.0238400246815348E8|1184.5000234154004|0.02991875377388485|0.060363710423265965|
|stddev |1.356743538196561E7|null               |288558.3518378498|157

+--------+--------+
|tx_fraud|   count|
+--------+--------+
|       1| 1406151|
|       0|45592832|
+--------+--------+


Fraud scenario distribution:


[Stage 78:=====================================================>  (23 + 1) / 24]

+-----------------+--------+
|tx_fraud_scenario|   count|
+-----------------+--------+
|                0|45592832|
|                1|   26226|
|                2| 1328968|
|                3|   50957|
+-----------------+--------+



## P.S. tranaction_id instead of transaction_id

Так как в сырых данных уже получили данную колонку, то, видимо, оставляем. Но потенциально это тоже ошибка